In [ ]:
import yfinance as yf

def get_stock_data(ticker: str) -> dict:
    
    stock = yf.Ticker(ticker)
    info = stock.info

    return {
        "ticker": ticker,
        "name": info.get("longName", ticker),
        "sector": info.get("sector", "N/A"),
        "industry": info.get("industry", "N/A"),
        "price": info.get("currentPrice", info.get("regularMarketPrice", 0)),
        "market_cap": info.get("marketCap", 0),
        "pe_ratio": info.get("trailingPE", "N/A"),
        "forward_pe": info.get("forwardPE", "N/A"),
        "peg_ratio": info.get("pegRatio", "N/A"),
        "revenue_growth": info.get("revenueGrowth", "N/A"),
        "profit_margin": info.get("profitMargins", "N/A"),
        "dividend_yield": info.get("dividendYield", 0),
        "52w_high": info.get("fiftyTwoWeekHigh", "N/A"),
        "52w_low": info.get("fiftyTwoWeekLow", "N/A"),
        "analyst_rating": info.get("recommendationKey", "N/A"),
        "target_price": info.get("targetMeanPrice", "N/A"),
        "description": info.get("longBusinessSummary", "")[:500],
    }

In [ ]:
import yfinance as yf
print(yf.__version__)

In [ ]:
import yfinance as yf

stock = yf.Ticker("NVDA")

print(stock.fast_info)

In [ ]:
stock = yf.Ticker("NVDA")

print(stock.history(period="5d"))

In [ ]:
import json

stock_data = get_stock_data(ticker="NVDA")

print(json.dumps(obj=stock_data, ensure_ascii=False, indent=2))

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

model = os.getenv('MODEL_NAME', 'gpt-5.5')

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

def analyze_stock(data: dict) -> str:
    llm = ChatOpenAI(model=model, temperature=0, extra_body={'enable_thinking': False})

    stock_info = "\n".join(f"{k}: {v}" for k, v in data.items() if k != "description")

    messages = [
      SystemMessage(content="你是一位金融分析师。请提供简洁的股票分析，涵盖以下方面：投资理由（2-3句话）、核心优势（3条）、核心风险（3条）、估值评估，以及投资建议（买入/持有/卖出，附简要理由）。字数制在300字以内。"),
      HumanMessage(content=f"请分析这只股票：\n{stock_info}\n\n公司简介：{data.get('description', 'N/A')}"),
  ]


    response = llm.invoke(messages)
    return response.content